# 🎯 K-Means Clustering - Implementación desde CERO

## Objetivos
- Entender clustering y K-Means
- Implementar K-Means desde cero
- Método del codo para elegir K
- Visualizar clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../..')
from utils.plot_utils import plot_clusters

## 1. Teoría de K-Means

K-Means es un algoritmo de **clustering no supervisado**.

### Objetivo:
Agrupar datos en K clusters minimizando la varianza intra-cluster.

### Algoritmo:
1. Inicializar K centroides aleatoriamente
2. **Asignar**: Cada punto al centroide más cercano
3. **Actualizar**: Recalcular centroides como media de puntos asignados
4. Repetir 2-3 hasta convergencia

### Función Objetivo:
$$J = \sum_{k=1}^{K} \sum_{x \in C_k} ||x - \mu_k||^2$$

Donde $\mu_k$ es el centroide del cluster k.

## 2. Implementación desde CERO

In [ ]:
class KMeans:
    """
    K-Means Clustering desde cero.
    """
    
    def __init__(self, n_clusters=3, max_iters=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_history = []
    
    def fit(self, X):
        """Entrena K-Means"""
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        X = np.array(X)
        n_samples, n_features = X.shape
        
        # Inicializar centroides: seleccionar K puntos aleatorios
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices]
        
        for iteration in range(self.max_iters):
            # Asignar cada punto al centroide más cercano
            self.labels = self._assign_clusters(X)
            
            # Guardar centroides anteriores
            old_centroids = self.centroids.copy()
            
            # Actualizar centroides
            self.centroids = self._update_centroids(X)
            
            # Calcular inercia
            inertia = self._calculate_inertia(X)
            self.inertia_history.append(inertia)
            
            # Verificar convergencia
            if np.allclose(old_centroids, self.centroids):
                print(f"Convergencia alcanzada en iteración {iteration+1}")
                break
        
        return self
    
    def _assign_clusters(self, X):
        """Asigna cada punto al cluster más cercano"""
        distances = np.zeros((X.shape[0], self.n_clusters))
        
        for i, centroid in enumerate(self.centroids):
            distances[:, i] = np.linalg.norm(X - centroid, axis=1)
        
        return np.argmin(distances, axis=1)
    
    def _update_centroids(self, X):
        """Actualiza centroides como media de puntos asignados"""
        new_centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for k in range(self.n_clusters):
            cluster_points = X[self.labels == k]
            if len(cluster_points) > 0:
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                # Si un cluster está vacío, reinicializar aleatoriamente
                new_centroids[k] = X[np.random.choice(X.shape[0])]
        
        return new_centroids
    
    def _calculate_inertia(self, X):
        """Calcula la inercia (suma de distancias al cuadrado)"""
        inertia = 0
        for k in range(self.n_clusters):
            cluster_points = X[self.labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - self.centroids[k]) ** 2)
        return inertia
    
    def predict(self, X):
        """Predice clusters para nuevos datos"""
        return self._assign_clusters(np.array(X))
    
    @property
    def inertia_(self):
        """Retorna la inercia final"""
        return self.inertia_history[-1] if self.inertia_history else None

## 3. Ejemplo de Clustering

In [ ]:
from sklearn.datasets import make_blobs

# Generar datos con 3 clusters bien separados
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                       cluster_std=0.6, random_state=42)

# Visualizar datos originales
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.6, edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos sin etiquetar')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Entrenar K-Means
kmeans = KMeans(n_clusters=3, max_iters=100, random_state=42)
kmeans.fit(X)

print(f"Inercia final: {kmeans.inertia_:.2f}")
print(f"Centroides:\n{kmeans.centroids}")

In [ ]:
# Visualizar clusters encontrados
plot_clusters(X, kmeans.labels, kmeans.centroids, 
             title='K-Means Clustering (K=3)')
plt.show()

In [ ]:
# Visualizar convergencia
plt.figure(figsize=(10, 6))
plt.plot(kmeans.inertia_history, linewidth=2, marker='o')
plt.xlabel('Iteración')
plt.ylabel('Inercia')
plt.title('Convergencia de K-Means')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Método del Codo (Elbow Method)

Para elegir el número óptimo de clusters K.

In [ ]:
# Probar diferentes valores de K
K_values = range(1, 11)
inertias = []

for k in K_values:
    kmeans_temp = KMeans(n_clusters=k, max_iters=100, random_state=42)
    kmeans_temp.fit(X)
    inertias.append(kmeans_temp.inertia_)

# Visualizar método del codo
plt.figure(figsize=(10, 6))
plt.plot(K_values, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo para determinar K óptimo')
plt.grid(True, alpha=0.3)
plt.axvline(x=3, color='r', linestyle='--', alpha=0.5, label='K óptimo')
plt.legend()
plt.show()

print("El 'codo' indica el K óptimo (donde la inercia deja de disminuir dramáticamente)")
print("En este caso, K=3 parece ser óptimo")

## 🎯 Ejercicio: Clustering de Datos Reales

Prueba K-Means en diferentes datasets.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Genera datos con make_blobs con diferentes parámetros
# 2. Usa el método del codo para encontrar K
# 3. Entrena K-Means con el K óptimo
# 4. Visualiza los clusters

print("Implementa tu solución aquí")

## 🎓 Resumen

- ✅ K-Means agrupa datos sin etiquetas
- ✅ Algoritmo iterativo: asignar → actualizar
- ✅ Método del codo para elegir K
- ✅ Sensible a inicialización (múltiples corridas)
- ✅ Asume clusters esféricos

### Próximo: Redes Neuronales desde Cero